## Step 1 — Verify GPU

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q --upgrade 'monai[all]==1.4.0'

# Install remaining packages
!pip install -q \
    pydicom \
    pynrrd \
    SimpleITK \
    nibabel \
    loguru \
    tqdm \
    scipy

# Verify
import importlib
required = {
    'monai'    : 'monai',
    'pydicom'  : 'pydicom',
    'nrrd'     : 'pynrrd',
    'SimpleITK': 'SimpleITK',
    'nibabel'  : 'nibabel',
    'loguru'   : 'loguru',
    'scipy'    : 'scipy',
}
all_ok = True
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — FAILED')
        all_ok = False

if all_ok:
    import monai
    print(f'\nAll packages ready. MONAI version: {monai.__version__}')
    if monai.__version__ < '1.4.0':
        print('WARNING: MONAI version is still < 1.4.0. Run Runtime → Restart session, then re-run this cell.')
    else:
        print('Version OK. Do NOT restart — proceed to Step 3.')
else:
    print('\nSome packages failed — run this cell again.')

# Step 3 - Pull project

In [ ]:
import os, sys
from pathlib import Path

GITHUB_REPO_URL = 'https://github.com/PsergiuT/AI-project.git'


PROJECT_DIR = Path('/content/aea-segmentation')

if PROJECT_DIR.exists():
    print(f'✓ Project already cloned — pulling latest changes...')
    os.system('cd /content/aea-segmentation && git pull')
else:
    ret = os.system(f'git clone {GITHUB_REPO_URL} /content/aea-segmentation')
    if ret != 0 or not PROJECT_DIR.exists():
        raise RuntimeError(
            'Git clone failed. Make sure the repository is PUBLIC.\n'
            'Go to GitHub → your repo → Settings → Change visibility → Make public\n'
            'Or use Option B (zip upload) instead.'
        )

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(str(PROJECT_DIR))
print(f'✓ Project ready at {PROJECT_DIR}')
print(f'  Files: {[f.name for f in PROJECT_DIR.iterdir() if f.is_file()]}')

# Step 4 - Setup storage bucket for the model

In [ ]:
import os, sys, json, re
from pathlib import Path

GCS_BUCKET = "aea-checkpoints-bucket_id"   # no way I am hardcoding my bucket here - place your own bucket
GCS_PATH   = f"gs://{GCS_BUCKET}"

PROJECT_DIR = Path("/content/aea-segmentation")
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(str(PROJECT_DIR))

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/splits",    exist_ok=True)

print("Restoring preprocessed NIfTI files from GCS...")
ret1 = os.system(f"gsutil -m cp -r {GCS_PATH}/data/processed ./data/")

print("Restoring split JSON files from GCS...")
ret2 = os.system(f"gsutil -m cp -r {GCS_PATH}/data/splits ./data/")

if ret1 != 0 or ret2 != 0:
    raise RuntimeError("GCS restore failed — check your bucket name and permissions.")

PROCESSED_DIR = str(PROJECT_DIR / "data" / "processed")

for split in ["train", "val", "test"]:
    json_path = Path(f"data/splits/{split}.json")
    if not json_path.exists():
        print(f"  WARNING: {split}.json not found")
        continue

    with open(json_path) as f:
        cases = json.load(f)

    for case in cases:
        # Replace whatever the old base path was with the Colab processed dir
        for key in ["image", "mask"]:
            old_path = case[key]
            filename = Path(old_path).name
            case_id  = case["case_id"]
            case[key] = str(Path(PROCESSED_DIR) / case_id / filename)

    with open(json_path, "w") as f:
        json.dump(cases, f, indent=2)

    print(f"  {split:>5}: {len(cases)} cases — paths updated")

print("
✓ Data restored and paths fixed. Ready to train.")


## Step 5 — Inspect a Sample Case


In [ ]:
import json
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

from config import SPLITS_DIR

# Load the first training case
with open(SPLITS_DIR / 'train.json') as f:
    train_manifest = json.load(f)

sample = train_manifest[0]
print(f"Case: {sample['case_id']}")

image = sitk.GetArrayFromImage(sitk.ReadImage(sample['image']))  # (Z, Y, X)
mask  = sitk.GetArrayFromImage(sitk.ReadImage(sample['mask']))

print(f'Image shape: {image.shape}, range: [{image.min():.0f}, {image.max():.0f}] HU')
print(f'Mask shape:  {mask.shape},  labels: {np.unique(mask)}')

# Find a slice containing AEA voxels
aea_slices = np.where(mask > 0)[0]
mid_slice  = aea_slices[len(aea_slices) // 2] if len(aea_slices) > 0 else image.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(image[mid_slice], cmap='gray', vmin=-1000, vmax=1000)
axes[0].set_title(f'CBCT — Slice {mid_slice} (axial)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(image[mid_slice], cmap='gray', vmin=-1000, vmax=1000)
overlay = np.ma.masked_where(mask[mid_slice] == 0, mask[mid_slice])
axes[1].imshow(overlay, cmap='bwr', alpha=0.6, vmin=0.5, vmax=2.5)
axes[1].set_title(f'AEA Overlay (Red=Left, Blue=Right) — Slice {mid_slice}', fontsize=13)
axes[1].axis('off')

plt.suptitle(f"Patient: {sample['case_id']}", fontsize=15)
plt.tight_layout()
plt.savefig('sample_case.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sample visualisation saved as sample_case.png')

## Step 6 — Train the Model


In [ ]:
import torch, os
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

total = torch.cuda.get_device_properties(0).total_memory
print(f'VRAM free: {(total - torch.cuda.memory_allocated(0)) / 1e9:.1f} GB / {total / 1e9:.1f} GB')

!PYTORCH_ALLOC_CONF=expandable_segments:True python src/train.py \
    --persistent \
    --num_workers 0

In [ ]:
import torch, os
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from config import SWINUNETR_DIR, TRAIN_CONFIG
last_ckpt = str(SWINUNETR_DIR / TRAIN_CONFIG['last_model_name'])
print(f'Resuming from: {last_ckpt}')

!PYTORCH_ALLOC_CONF=expandable_segments:True python src/train.py \
    --persistent \
    --num_workers 2 \
    --resume "{last_ckpt}"

## Step 7 — Plot Training History

In [ ]:
import json
import matplotlib.pyplot as plt
from config import LOGS_DIR

with open(LOGS_DIR / 'training_history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Training Loss (DiceCE)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

val_epochs = list(range(4, len(history['train_loss']) + 1, 5))[:len(history['val_dice'])]
axes[1].plot(val_epochs, history['val_dice'], color='green', marker='o', markersize=3)
axes[1].set_title('Validation Dice Score', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(val_epochs, history['val_hd95'], color='crimson', marker='o', markersize=3)
axes[2].set_title('Validation HD95 (mm)', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('HD95 (mm)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('SwinUNETR Training Progress — AEA Segmentation', fontsize=14)
plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()
print('Training history plot saved as training_history.png')

## Step 8 — Evaluate on Test Set

In [ ]:
import torch
import json
from config import SWINUNETR_DIR, TRAIN_CONFIG, SPLITS_DIR, INFERENCE_CONFIG
from src.dataset import get_dataloader
from src.evaluate import evaluate_model
from src.train import build_model
from src.utils import save_json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best model
best_ckpt = SWINUNETR_DIR / TRAIN_CONFIG['best_model_name']
model = build_model(device, pretrained=False)  # Architecture only, no pretrained weights
checkpoint = torch.load(str(best_ckpt), map_location=device)
model.load_state_dict(checkpoint['model'])
print(f"Loaded best model (val Dice: {checkpoint['best_dice']:.4f})")

# Load test set
with open(SPLITS_DIR / 'test.json') as f:
    test_manifest = json.load(f)
test_loader = get_dataloader(test_manifest, mode='test', num_workers=2)

# Evaluate
print('\nRunning test set evaluation...')
test_results = evaluate_model(
    model      = model,
    dataloader = test_loader,
    device     = device,
    roi_size   = INFERENCE_CONFIG['roi_size'],
    sw_batch   = INFERENCE_CONFIG['sw_batch_size'],
)

# Save and display results
save_json(test_results, LOGS_DIR / 'test_results.json')

print('\n' + '='*50)
print('FINAL TEST SET RESULTS')
print('='*50)
print(f"{'Metric':<20} {'AEA Left':>10} {'AEA Right':>10} {'Mean':>10}")
print('-'*50)
print(f"{'Dice (DSC)':<20} {test_results['dice_aeal']:>10.4f} {test_results['dice_aear']:>10.4f} {test_results['dice_mean']:>10.4f}")
print(f"{'IoU (Jaccard)':<20} {test_results['iou_aeal']:>10.4f} {test_results['iou_aear']:>10.4f} {test_results['iou_mean']:>10.4f}")
print(f"{'HD95 (mm)':<20} {test_results['hd95_aeal']:>10.4f} {test_results['hd95_aear']:>10.4f} {test_results['hd95_mean']:>10.4f}")
print('='*50)

## Step 9 — Visualise Predictions on Test Cases

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import SimpleITK as sitk
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

from config import INFERENCE_CONFIG
from src.postprocess import full_postprocess

model.eval()
post_pred = AsDiscrete(argmax=True)

# Visualise first 3 test cases
n_show = min(3, len(test_manifest))

for i, case in enumerate(test_manifest[:n_show]):
    image_np = sitk.GetArrayFromImage(sitk.ReadImage(case['image']))
    mask_np  = sitk.GetArrayFromImage(sitk.ReadImage(case['mask']))

    # Run inference on a single case (reuse test_loader batch)
    from src.dataset import get_transforms
    from monai.data import CacheDataset, DataLoader

    single_loader = DataLoader(
        CacheDataset([case], transform=get_transforms('test'), cache_rate=1.0),
        batch_size=1
    )
    batch = next(iter(single_loader))
    with torch.no_grad():
        logits = sliding_window_inference(
            batch['image'].to(device),
            roi_size      = INFERENCE_CONFIG['roi_size'],
            sw_batch_size = INFERENCE_CONFIG['sw_batch_size'],
            predictor     = model,
            overlap       = 0.5,
        )
    pred = post_pred(logits[0]).cpu().numpy().squeeze()  # (H, W, D)
    pred = full_postprocess(pred)

    # Find best slice
    aea_z = np.where((pred > 0) | (mask_np > 0))[0]
    z_mid = aea_z[len(aea_z) // 2] if len(aea_z) > 0 else image_np.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    axes[0].set_title('CBCT', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    gt_overlay = np.ma.masked_where(mask_np[z_mid] == 0, mask_np[z_mid])
    axes[1].imshow(gt_overlay, cmap='bwr', alpha=0.7, vmin=0.5, vmax=2.5)
    axes[1].set_title('Ground Truth', fontsize=11)
    axes[1].axis('off')

    axes[2].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    pred_overlay = np.ma.masked_where(pred[z_mid] == 0, pred[z_mid])
    axes[2].imshow(pred_overlay, cmap='bwr', alpha=0.7, vmin=0.5, vmax=2.5)
    axes[2].set_title('Prediction (post-processed)', fontsize=11)
    axes[2].axis('off')

    plt.suptitle(f"Test Case: {case['case_id']} — Slice {z_mid}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"prediction_{case['case_id']}.png", dpi=120, bbox_inches='tight')
    plt.show()

print(f'Visualisation complete for {n_show} test cases.')

## Step 10 — Download Trained Model to Your Computer


In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import files
from config import SWINUNETR_DIR, LOGS_DIR, TRAIN_CONFIG

# Bundle model + logs into a single zip for download
bundle_path = Path('/content/aea_model_bundle.zip')

with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    best_ckpt = SWINUNETR_DIR / TRAIN_CONFIG['best_model_name']
    if best_ckpt.exists():
        zf.write(best_ckpt, f'models/swinunetr/{best_ckpt.name}')
        print(f'✓ Added model checkpoint ({best_ckpt.stat().st_size / 1e6:.0f} MB)')
    else:
        print('✗ Best checkpoint not found — did training complete?')

    # Training history + test results
    for log_file in ['training_history.json', 'test_results.json']:
        p = LOGS_DIR / log_file
        if p.exists():
            zf.write(p, f'logs/{log_file}')
            print(f'✓ Added {log_file}')

bundle_size = bundle_path.stat().st_size / 1e6
print(f'\nBundle ready: {bundle_path.name} ({bundle_size:.0f} MB)')
print('Starting download to your computer...')

# This triggers a browser download
files.download(str(bundle_path))

print('\nDone! After download:')
print('  1. Extract aea_model_bundle.zip')
print('  2. Copy models/swinunetr/swinunetr_best.pth into your local aea-segmentation/models/swinunetr/')
print('  3. Run: python run.py')